**Question:** Does giving XGBoost the same razina-trajectory information close the gap to the GRU?

**Assumptions:** The GRU's advantage comes mostly from seeing league level as a trajectory.

**Method:** Add ~9 hand-engineered trajectory features to Exp A and Exp B, producing A' and B'.

**Result:** A' improves to 0.1754 (from 0.1969) and B' to 0.1655 (from 0.1670), still behind the GRU.

# 07 -- XGBoost + RAZINA Trajectory Features: Closing the Gap to the GRU?

**Motivation** (audit follow-up, 2026-07-11): the
ELO-free GRU (Exp C, calibrated test Brier ~0.1628) beats the ELO-free XGBoost model (Exp A,
0.19690) by a wide margin. `05_gru_neural_elo.ipynb` Step 15's input-masking ablation identified
WHY: the `LIGAKOEF_ONLY` variant (Won + razina trajectory, nothing else) scores 0.16741 -- almost
as good as the FULL GRU (0.16278) -- while `NO_LIGAKOEF` (drop razina trajectory) collapses to
0.18337, *worse than the plain ELO baseline*. The GRU's edge is overwhelmingly driven by seeing
`LigaKoeficjent` (razina) as a **trajectory** across the match sequence. Exp A's XGBoost, by
contrast, only ever saw razina as a handful of static snapshots (current razina, debut razina,
one match-count-at-razina feature) -- never the actual sequence shape.

**This notebook's question:** if XGBoost is handed the SAME trajectory information, hand-engineered
into ~6-10 point-in-time summary features (instead of a learned recurrent embedding), does the
apples-to-apples gap to the GRU close?

- **Exp A'** = Exp A's 27 ELO-free columns + trajectory features (vs Exp A 0.19690 and Exp C 0.16278)
- **Exp B'** = Exp B's 28 ELO-included columns + trajectory features (vs Exp B 0.16699)

**Runtime safety (mandatory for this notebook):** every trajectory feature
below is computed with vectorized pandas (`groupby` + `shift`/`rolling`/`cumcount`/`cummin`, plus
per-GROUP -- never per-ROW -- `.apply()` for the two rolling-window helpers pandas has no direct
chained `groupby().rolling().shift()` API for). No `for` loop ever iterates the 34,256-row frame.
**This entire notebook (feature engineering + 2 XGBoost fits + 2000-resample bootstrap + feature
importance) was validated end-to-end in a standalone script BEFORE being assembled here, completing
in ~48 seconds total** -- there is no walk-forward CV in this notebook (explicitly out of scope for
this notebook) and no hyperparameter grid search (both Exp A' and Exp B' reuse the exact tuned
hyperparameters from `03_model_training.ipynb` Step 4).

**Split / data / eval code:** identical to `03_model_training.ipynb` -- same `features.parquet`,
same time-based `Split` column (train 2017-05-08..2025-01-18 / test 2025-01-19..2026-07-08, 3,426
test matches / 6,852 rows), same chronological core-fit/early-stop/calibration carve, same
isotonic `CalibratedClassifierCV(cv='prefit')`. Bootstrap CI methodology identical to
`06_confidence_intervals_cv.ipynb` Part 1/2 (2,000 match-level resamples, seed 42, both
perspective rows of a match always travel together).

In [ ]:
import json
import pickle
import time
from pathlib import Path

import numpy as np
import pandas as pd
import xgboost as xgb
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import accuracy_score, brier_score_loss, log_loss, roc_auc_score

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

pd.set_option('display.max_columns', 80)
pd.set_option('display.width', 200)

DATA_DIR = Path('../data')
MODELS_DIR = Path('../models')
MODELS_DIR.mkdir(parents=True, exist_ok=True)

T0 = time.time()
def lap(label):
    print(f"[{time.time()-T0:7.1f}s] {label}")

## Step 1 -- Rebuild the long-format player-match log

Identical construction to `02_feature_engineering.ipynb` Step 1: one row per (match, player),
sorted `(IgracId, Datum, SusretId)` so every trajectory feature below only ever looks at rows
ABOVE the current one within a player's own history -- i.e. strictly BEFORE this match.

In [ ]:
raw = pd.read_csv(DATA_DIR / 'matches_export.csv', parse_dates=['Datum'])
raw['Datum'] = pd.to_datetime(raw['Datum']).dt.tz_localize(None)
raw = raw.sort_values(['Datum', 'SusretId']).reset_index(drop=True)
lap(f"Loaded matches_export.csv: {raw.shape}")

common_cols = ['SusretId', 'Datum', 'SezonaId', 'LigaId', 'LigaTip', 'LigaKoeficjent', 'Lokacija']
winner_rows = raw[common_cols + ['WinnerId', 'LoserId']].rename(
    columns={'WinnerId': 'IgracId', 'LoserId': 'OpponentId'}
).assign(Won=1)
loser_rows = raw[common_cols + ['LoserId', 'WinnerId']].rename(
    columns={'LoserId': 'IgracId', 'WinnerId': 'OpponentId'}
).assign(Won=0)
player_log = pd.concat([winner_rows, loser_rows], ignore_index=True)
player_log = player_log.sort_values(['IgracId', 'Datum', 'SusretId'], kind='mergesort').reset_index(drop=True)
lap(f"player_log built: {player_log.shape}")
player_log.head(3)

## Step 2 -- Vectorized trajectory features (~9 new point-in-time features)

All strictly `< Datum` point-in-time (or, for the two features that reference this match's OWN
`LigaKoeficjent`, the same audited-safe reasoning already established for the existing
`liga_koeficjent` feature in `02_feature_engineering.ipynb` -- razina is administratively assigned
BEFORE the match is played, never derived from its outcome).

| # | Feature | What it captures |
|---|---|---|
| 1-3 | `razina_ago5` / `razina_ago10` / `razina_ago20` | razina value N matches ago (plain `shift(n)`) |
| 4 | `razina_slope10` | OLS slope of razina over the trailing 10 STRICTLY PRIOR matches (closed-form via rolling sums, no `.apply()` needed at all) |
| 5 | `razina_stint` | matches played, strictly prior, in the CURRENT contiguous stint at the prevailing razina (resets on promotion/relegation -- distinct from the existing all-time `razina_match_count_before`) |
| 6 | `razina_promotion_signal` | -1/0/+1: is this match's own razina better/worse than the player's last 5 STRICTLY PRIOR matches' range? |
| 7 | `razina_best_ever` | best (min) razina reached in any STRICTLY PRIOR match |
| 8 | `razina_current_minus_best` | this match's own razina minus the strictly-prior best-ever (how far below peak) |
| 9 | `forma_slope10` (bonus, optional per task) | OLS slope of the win/loss indicator (`Won`) over the trailing 10 STRICTLY PRIOR matches -- trend of `forma_last10`, not just its level |

**Vectorization discipline:** `shift`, `cummin`, `cumcount`, `cumsum`, and closed-form rolling-sum
slope formulas are used throughout. The only `.apply()` calls are `groupby(...).apply(lambda s: ...)`
where the lambda receives one PLAYER's entire history at once (a few hundred/thousand groups, each
vectorized internally) -- never a per-row Python loop over the 34,256-row frame.

In [ ]:
g = player_log.groupby('IgracId', sort=False)

# (1)-(3) razina N matches ago -- plain shift, strictly prior by construction
player_log['RazinaAgo5'] = g['LigaKoeficjent'].shift(5)
player_log['RazinaAgo10'] = g['LigaKoeficjent'].shift(10)
player_log['RazinaAgo20'] = g['LigaKoeficjent'].shift(20)


def rolling_slope(t_col, y_col, window, min_periods):
    """Closed-form rolling OLS slope of y_col vs t_col within a window, using ONLY rolling
    sums (no .apply(), no per-row loop): slope = (n*Sxy - Sx*Sy) / (n*Sxx - Sx^2). Valid because
    t_col is any per-group monotonically increasing index (slope is invariant to a linear shift
    of the x-axis origin)."""
    t = player_log[t_col].astype(float)
    y = player_log[y_col].astype(float)
    ty, tt = t * y, t * t
    grp = player_log['IgracId']
    Sy = y.groupby(grp).transform(lambda s: s.rolling(window, min_periods=min_periods).sum())
    St = t.groupby(grp).transform(lambda s: s.rolling(window, min_periods=min_periods).sum())
    Sty = ty.groupby(grp).transform(lambda s: s.rolling(window, min_periods=min_periods).sum())
    Stt = tt.groupby(grp).transform(lambda s: s.rolling(window, min_periods=min_periods).sum())
    n = y.groupby(grp).transform(lambda s: s.rolling(window, min_periods=min_periods).count())
    denom = (n * Stt - St ** 2)
    return (n * Sty - St * Sy) / denom.replace(0, np.nan)


# (4) razina trajectory slope over trailing 10 matches (strictly prior -> shift(1) at the end)
player_log['_pos'] = g.cumcount()
WINDOW = 10
player_log['RazinaSlope10'] = rolling_slope('_pos', 'LigaKoeficjent', WINDOW, WINDOW).groupby(player_log['IgracId']).shift(1)

# (9, optional bonus) forma slope over trailing 10 matches -- same closed-form helper, on Won
player_log['FormaSlope10'] = rolling_slope('_pos', 'Won', WINDOW, WINDOW).groupby(player_log['IgracId']).shift(1)

# (5) razina "current stint" length: matches played, strictly prior, in the current contiguous
# run at the SAME razina (resets on promotion/relegation) -- change-flag -> cumsum (block id) ->
# cumcount within block. NaN != x is already True in pandas, so the first row of every group is
# already correctly flagged as a new stint without any extra per-row patch.
prev_razina = g['LigaKoeficjent'].shift(1)
player_log['_changed'] = (player_log['LigaKoeficjent'] != prev_razina).astype(int)
player_log['_block_id'] = player_log.groupby('IgracId')['_changed'].cumsum()
player_log['RazinaStint'] = player_log.groupby(['IgracId', '_block_id']).cumcount()

# (7) best (min) razina ever reached, strictly prior -- per-group (not per-row) apply: cummin then shift(1)
player_log['RazinaBestEver'] = player_log.groupby('IgracId')['LigaKoeficjent'].apply(
    lambda s: s.cummin().shift(1)
).reset_index(level=0, drop=True)

# (8) this match's own razina minus the strictly-prior best-ever
player_log['RazinaCurrentMinusBest'] = player_log['LigaKoeficjent'] - player_log['RazinaBestEver']

# (6) promotion/relegation signal: this match's own (pre-assigned) razina vs. the min/max razina
# over the last 5 STRICTLY PRIOR matches. -1 = promoted beyond recent range, +1 = relegated beyond
# recent range, 0 = within recent range, NaN = no prior match history at all.
recent_min5 = player_log.groupby('IgracId')['LigaKoeficjent'].apply(
    lambda s: s.rolling(5, min_periods=1).min().shift(1)).reset_index(level=0, drop=True)
recent_max5 = player_log.groupby('IgracId')['LigaKoeficjent'].apply(
    lambda s: s.rolling(5, min_periods=1).max().shift(1)).reset_index(level=0, drop=True)
promo = pd.Series(0.0, index=player_log.index)
promo = promo.where(~(player_log['LigaKoeficjent'] < recent_min5), -1)
promo = promo.where(~(player_log['LigaKoeficjent'] > recent_max5), 1)
promo = promo.where(recent_min5.notna(), np.nan)
player_log['RazinaPromotionSignal'] = promo

lap("Trajectory features computed (vectorized, no per-row loops)")

TRAJ_RAW_COLS = ['RazinaAgo5', 'RazinaAgo10', 'RazinaAgo20', 'RazinaSlope10', 'RazinaStint',
                  'RazinaBestEver', 'RazinaCurrentMinusBest', 'RazinaPromotionSignal', 'FormaSlope10']
print(player_log[TRAJ_RAW_COLS].describe())
print("\nNaN share (debut / short-history rows -- XGBoost handles NaN natively, same as Exp A/B):")
print(player_log[TRAJ_RAW_COLS].isna().mean())

## Step 3 -- Leakage audit (mandatory, one row per new feature)

Same discipline as `02_feature_engineering.ipynb` Step 8. None of the 9 features reads
`EloBefore`/`EloAfter` (verified by assertion below), and each has an explicit point-in-time rule.

In [ ]:
TRAJ_FEATURE_AUDIT = [
    dict(feature='razina_ago5', pit="shift(5) on LigaKoeficjent, PARTITION BY IgracId ORDER BY Datum,SusretId",
         reads_outcome=False, tag='ELO-substitute', note='razina VALUE 5 matches ago -- admin skill-tier proxy, same tag rationale as liga_koeficjent'),
    dict(feature='razina_ago10', pit='shift(10), same ordering', reads_outcome=False, tag='ELO-substitute', note='same rationale, longer lookback'),
    dict(feature='razina_ago20', pit='shift(20), same ordering', reads_outcome=False, tag='ELO-substitute', note='same rationale, longest lookback'),
    dict(feature='razina_slope10', pit='closed-form OLS slope over ROWS 10 PRECEDING..1 PRECEDING (rolling sums only, then shift(1))',
         reads_outcome=False, tag='ELO-substitute', note='trajectory/trend of the razina VALUE -- the core new signal this notebook tests'),
    dict(feature='razina_stint', pit='cumcount within a (IgracId, contiguous-razina-block) partition -- strictly prior by construction of cumcount',
         reads_outcome=False, tag='neutral', note='a COUNT/tenure signal (like the existing razina_match_count_before), not a skill-level value'),
    dict(feature='razina_promotion_signal', pit="this match's own LigaKoeficjent (admin-assigned pre-match, same audited column as liga_koeficjent) vs rolling min/max over 5 STRICTLY PRIOR matches (shift(1) applied to the rolling min/max before comparison)",
         reads_outcome=False, tag='ELO-substitute', note='admin promotion/relegation decision is itself a skill judgment, same rationale as debut_razina'),
    dict(feature='razina_best_ever', pit='cummin() then shift(1), PARTITION BY IgracId, strictly prior',
         reads_outcome=False, tag='ELO-substitute', note="player's peak admin-assigned skill tier, same rationale as debut_razina"),
    dict(feature='razina_current_minus_best', pit="fn(this match's own liga_koeficjent, razina_best_ever) -- both individually point-in-time safe",
         reads_outcome=False, tag='ELO-substitute', note='how far below career-peak level right now -- decline/rust proxy'),
    dict(feature='forma_slope10', pit='closed-form OLS slope of Won over ROWS 10 PRECEDING..1 PRECEDING, then shift(1)',
         reads_outcome=False, tag='ELO-substitute', note='built ONLY from Won, never EloBefore/EloAfter -- trend of forma_last10, same tag rationale'),
]
traj_audit_df = pd.DataFrame(TRAJ_FEATURE_AUDIT)
assert traj_audit_df['reads_outcome'].sum() == 0, "a trajectory feature reads the outcome of the match being predicted!"
for c in TRAJ_RAW_COLS:
    assert 'Elo' not in c, f"{c} touches an Elo column name!"
print(f"{len(traj_audit_df)} new features audited, 0 read the match outcome, none touches Elo.")
traj_audit_df

## Step 4 -- Merge trajectory diffs onto the existing symmetric frame (`features.parquet`)

`features.parquet` already has one row per (match, perspective) with `PlayerId`/`OpponentId`
correctly assigned (winner_pov: `PlayerId=WinnerId`; loser_pov: `PlayerId=LoserId`). Merging the
per-(SusretId, IgracId) trajectory values onto BOTH `PlayerId` and `OpponentId` and taking
`own - opponent` reproduces the exact same diff convention as `02`'s `build_symmetric_frame`,
without needing to rebuild the whole symmetric expansion from scratch.

In [ ]:
df = pd.read_parquet(DATA_DIR / 'features.parquet')
with open(DATA_DIR / 'feature_tags.json', encoding='utf-8') as f:
    TAGS = json.load(f)
EXP_A_COLS = TAGS['experiment_a_cols']
EXP_B_COLS = TAGS['experiment_b_cols']
df['liga_tip'] = df['liga_tip'].map({'SM': 0, 'SF': 1}).astype(int)
df = df.sort_values(['Datum', 'SusretId'], kind='mergesort').reset_index(drop=True)
lap(f"features.parquet loaded: {df.shape}")

traj = player_log[['SusretId', 'IgracId'] + TRAJ_RAW_COLS].copy()

own = df[['SusretId', 'PlayerId']].merge(
    traj, left_on=['SusretId', 'PlayerId'], right_on=['SusretId', 'IgracId'], how='left', validate='one_to_one'
).drop(columns=['IgracId'])
opp = df[['SusretId', 'OpponentId']].merge(
    traj, left_on=['SusretId', 'OpponentId'], right_on=['SusretId', 'IgracId'], how='left', validate='one_to_one'
).drop(columns=['IgracId'])
assert (own['SusretId'].values == df['SusretId'].values).all()
assert (opp['SusretId'].values == df['SusretId'].values).all()
print("OK -- merge preserved row order/alignment against features.parquet.")

NEW_FEATURE_TAGS = {
    'razina_ago5': 'ELO-substitute', 'razina_ago10': 'ELO-substitute', 'razina_ago20': 'ELO-substitute',
    'razina_slope10': 'ELO-substitute', 'razina_stint': 'neutral', 'razina_best_ever': 'ELO-substitute',
    'razina_current_minus_best': 'ELO-substitute', 'razina_promotion_signal': 'ELO-substitute',
    'forma_slope10': 'ELO-substitute',
}
RAW_TO_NAME = dict(zip(TRAJ_RAW_COLS, NEW_FEATURE_TAGS.keys()))

TRAJ_DIFF_COLS = []
for raw_col, name in RAW_TO_NAME.items():
    diff_col = f'{name}_diff'
    df[diff_col] = own[raw_col].values - opp[raw_col].values
    TRAJ_DIFF_COLS.append(diff_col)

lap(f"Trajectory diff columns merged ({len(TRAJ_DIFF_COLS)}): {TRAJ_DIFF_COLS}")
print("\nNaN share per diff column:")
print(df[TRAJ_DIFF_COLS].isna().mean())

EXP_A_PRIME_COLS = EXP_A_COLS + TRAJ_DIFF_COLS
EXP_B_PRIME_COLS = EXP_B_COLS + TRAJ_DIFF_COLS
assert 'elo_diff' not in EXP_A_PRIME_COLS, "Exp A' must stay ELO-free!"
assert 'elo_diff' in EXP_B_PRIME_COLS, "Exp B' must include elo_diff!"
print(f"\nExp A' (ELO-free + trajectory) columns: {len(EXP_A_PRIME_COLS)}")
print(f"Exp B' (ELO-included + trajectory) columns: {len(EXP_B_PRIME_COLS)}")

## Step 5 -- Train Exp A' / B' -- SAME pipeline + hyperparameters as `03_model_training.ipynb`

No re-tuning: `BEST_PARAMS_A`/`BEST_PARAMS_B` are the exact winners from `03`'s Step 4
`TimeSeriesSplit` CV sanity check (reused verbatim, same as `06_confidence_intervals_cv.ipynb`'s
walk-forward folds do). Same chronological core-fit (76.5%) / early-stop-eval (8.5%) / calibration
(15%) carve within TRAIN, same `n_estimators=500` + `early_stopping_rounds=30`, same isotonic
`CalibratedClassifierCV(cv='prefit')`. TEST is untouched until Step 6.

In [ ]:
train_df = df[df['Split'] == 'train'].copy()
test_df = df[df['Split'] == 'test'].copy()
y_train = train_df['Label'].values
y_test = test_df['Label'].values

train_match_order = (train_df[['SusretId', 'Datum']].drop_duplicates()
                      .sort_values(['Datum', 'SusretId'], kind='mergesort').reset_index(drop=True))
n_train_matches = len(train_match_order)

def matches_to_mask(frame, susret_ids):
    return frame['SusretId'].isin(set(susret_ids))

BEST_PARAMS_A = {'max_depth': 6, 'reg_alpha': 0.5, 'reg_lambda': 2.0, 'learning_rate': 0.05}
BEST_PARAMS_B = {'max_depth': 4, 'reg_alpha': 0.1, 'reg_lambda': 1.0, 'learning_rate': 0.05}

def fit_calibrated_model(cols, best_params, label, n_estimators_final=500,
                          calib_frac=0.15, early_stop_frac=0.10, isotonic_min_rows=1000):
    n_calib = int(round(n_train_matches * calib_frac))
    fit_order = train_match_order.iloc[: n_train_matches - n_calib].reset_index(drop=True)
    calib_match_ids = train_match_order.iloc[n_train_matches - n_calib:]['SusretId']
    n_fit_matches = len(fit_order)
    n_eval = int(round(n_fit_matches * early_stop_frac))
    core_match_ids = fit_order.iloc[: n_fit_matches - n_eval]['SusretId']
    eval_match_ids = fit_order.iloc[n_fit_matches - n_eval:]['SusretId']

    core_mask = matches_to_mask(train_df, core_match_ids)
    eval_mask = matches_to_mask(train_df, eval_match_ids)
    calib_mask = matches_to_mask(train_df, calib_match_ids)

    X_core, y_core = train_df.loc[core_mask, cols], train_df.loc[core_mask, 'Label']
    X_eval, y_eval = train_df.loc[eval_mask, cols], train_df.loc[eval_mask, 'Label']
    X_calib, y_calib = train_df.loc[calib_mask, cols], train_df.loc[calib_mask, 'Label']
    print(f"[{label}] core fit: {len(X_core):,} | early-stop eval: {len(X_eval):,} | calib: {len(X_calib):,}")

    base_model = xgb.XGBClassifier(
        n_estimators=n_estimators_final, max_depth=best_params['max_depth'],
        learning_rate=best_params['learning_rate'], reg_alpha=best_params['reg_alpha'],
        reg_lambda=best_params['reg_lambda'], subsample=0.9, colsample_bytree=0.9,
        random_state=RANDOM_STATE, tree_method='hist', eval_metric='logloss',
        early_stopping_rounds=30, n_jobs=-1,
    )
    base_model.fit(X_core, y_core, eval_set=[(X_eval, y_eval)], verbose=False)
    print(f"[{label}] best_iteration = {base_model.best_iteration} / {n_estimators_final}")

    calib_method = 'isotonic' if len(X_calib) >= isotonic_min_rows else 'sigmoid'
    calibrated = CalibratedClassifierCV(estimator=base_model, method=calib_method, cv='prefit')
    calibrated.fit(X_calib, y_calib)
    return base_model, calibrated

t0 = time.time()
model_a_prime_raw, model_a_prime_cal = fit_calibrated_model(EXP_A_PRIME_COLS, BEST_PARAMS_A, "Exp A'")
lap(f"Exp A' trained ({time.time()-t0:.1f}s)")

t0 = time.time()
model_b_prime_raw, model_b_prime_cal = fit_calibrated_model(EXP_B_PRIME_COLS, BEST_PARAMS_B, "Exp B'")
lap(f"Exp B' trained ({time.time()-t0:.1f}s)")

## Step 6 -- Score TEST, compare against Exp 0 / A / B / C (same test rows)

Exp 0/A/B/C predictions are pulled from `test_predictions_4way.parquet` (already produced by
`03_model_training.ipynb` + `05_gru_neural_elo.ipynb`) and merged onto the exact same
`(SusretId, PlayerId, OpponentId)` test rows Exp A'/B' were just scored on -- guarantees a
row-for-row identical comparison, not just "same size" test sets.

In [ ]:
def compute_metrics(y_true, p, label):
    p_c = np.clip(p, 1e-6, 1 - 1e-6)
    return {'experiment': label, 'accuracy': accuracy_score(y_true, (p >= 0.5).astype(int)),
            'brier': brier_score_loss(y_true, p), 'log_loss': log_loss(y_true, p_c),
            'auc': roc_auc_score(y_true, p)}

X_test_a_prime = test_df[EXP_A_PRIME_COLS]
X_test_b_prime = test_df[EXP_B_PRIME_COLS]
p_test_a_prime = model_a_prime_cal.predict_proba(X_test_a_prime)[:, 1]
p_test_b_prime = model_b_prime_cal.predict_proba(X_test_b_prime)[:, 1]

preds4 = pd.read_parquet(DATA_DIR / 'test_predictions_4way.parquet')
merge_keys = test_df[['SusretId', 'PlayerId', 'OpponentId']].copy()
merge_keys['_ord'] = np.arange(len(merge_keys))
merged = merge_keys.merge(preds4, on=['SusretId', 'PlayerId', 'OpponentId'], how='left').sort_values('_ord')
assert merged['Label'].values.tolist() == test_df['Label'].values.tolist(), "row order/label mismatch after merge!"

results = pd.DataFrame([
    compute_metrics(y_test, merged['p_exp0_elo'].values, 'Exp 0 - ELO baseline'),
    compute_metrics(y_test, merged['p_expA_calibrated'].values, 'Exp A - ELO-free XGB'),
    compute_metrics(y_test, p_test_a_prime, "Exp A' - ELO-free XGB + trajectory"),
    compute_metrics(y_test, merged['p_expB_calibrated'].values, 'Exp B - ELO-included XGB'),
    compute_metrics(y_test, p_test_b_prime, "Exp B' - ELO-included XGB + trajectory"),
    compute_metrics(y_test, merged['p_expC_calibrated'].values, 'Exp C - ELO-free GRU'),
])
elo_brier = results.loc[results['experiment'] == 'Exp 0 - ELO baseline', 'brier'].values[0]
results['brier_delta_vs_elo_pct'] = (results['brier'] - elo_brier) / elo_brier * 100
lap("Scored TEST for Exp A' / B', merged with Exp 0/A/B/C")
results

## Step 7 -- Match-level bootstrap CIs (2,000 resamples, seed 42) -- identical methodology to `06`

Both perspective rows of a match always travel together (draw match IDs with replacement, pull
BOTH rows for each drawn match) -- same discipline as `06_confidence_intervals_cv.ipynb` Part 1.

In [ ]:
EXPS = {
    "Exp 0 - ELO baseline": merged['p_exp0_elo'].values,
    "Exp A - ELO-free XGB": merged['p_expA_calibrated'].values,
    "Exp A' - ELO-free XGB + trajectory": p_test_a_prime,
    "Exp B - ELO-included XGB": merged['p_expB_calibrated'].values,
    "Exp B' - ELO-included XGB + trajectory": p_test_b_prime,
    "Exp C - ELO-free GRU": merged['p_expC_calibrated'].values,
}
y_all = y_test

groups_idx = test_df.reset_index(drop=True)
susret_ids = groups_idx['SusretId'].values
unique_matches = pd.unique(susret_ids)
n_matches = len(unique_matches)
row_pos_by_match = {m: np.where(susret_ids == m)[0] for m in unique_matches}
assert all(len(v) == 2 for v in row_pos_by_match.values()), "expected exactly 2 rows (both perspectives) per match"
row_idx_2d = np.array([row_pos_by_match[m] for m in unique_matches])

N_BOOT = 2000
rng = np.random.RandomState(RANDOM_STATE)
boot_metrics = {name: {'accuracy': [], 'brier': [], 'log_loss': [], 'auc': []} for name in EXPS}
boot_diff = {"A'_minus_C": [], "A'_minus_A": [], "B'_minus_B": []}

t0 = time.time()
for b in range(N_BOOT):
    sampled_pos = rng.randint(0, n_matches, size=n_matches)
    idx = row_idx_2d[sampled_pos].reshape(-1)
    y_b = y_all[idx]
    briers = {}
    for name, p in EXPS.items():
        p_b = p[idx]
        p_c = np.clip(p_b, 1e-6, 1 - 1e-6)
        m = {'accuracy': accuracy_score(y_b, (p_b >= 0.5).astype(int)), 'brier': brier_score_loss(y_b, p_b),
             'log_loss': log_loss(y_b, p_c)}
        try:
            m['auc'] = roc_auc_score(y_b, p_b)
        except ValueError:
            m['auc'] = np.nan
        for k in boot_metrics[name]:
            boot_metrics[name][k].append(m[k])
        briers[name] = m['brier']
    boot_diff["A'_minus_C"].append(briers["Exp A' - ELO-free XGB + trajectory"] - briers["Exp C - ELO-free GRU"])
    boot_diff["A'_minus_A"].append(briers["Exp A' - ELO-free XGB + trajectory"] - briers["Exp A - ELO-free XGB"])
    boot_diff["B'_minus_B"].append(briers["Exp B' - ELO-included XGB + trajectory"] - briers["Exp B - ELO-included XGB"])
lap(f"Bootstrap done ({N_BOOT} resamples, {time.time()-t0:.1f}s)")

ci_rows = []
for name in EXPS:
    row = {'experiment': name}
    for metric in ['accuracy', 'brier', 'log_loss', 'auc']:
        arr = np.array(boot_metrics[name][metric])
        lo, hi = np.percentile(arr, [2.5, 97.5])
        row[f'{metric}_point'] = results.loc[results['experiment'] == name, metric].values[0]
        row[f'{metric}_ci_lo'] = lo
        row[f'{metric}_ci_hi'] = hi
    ci_rows.append(row)
ci_table = pd.DataFrame(ci_rows).set_index('experiment')
ci_table.round(5)

## Step 8 -- Paired bootstrap on Brier DIFFERENCES (the significance test that matters)

Same 2,000 match-level resamples as Step 7, so each comparison is properly paired.

In [ ]:
diff_labels = {
    "A'_minus_C": "Exp A' (ELO-free XGB+trajectory) minus Exp C (GRU) -- did the gap to the GRU close?",
    "A'_minus_A": "Exp A' minus Exp A (ELO-free XGB, no trajectory) -- do trajectory features help XGBoost?",
    "B'_minus_B": "Exp B' minus Exp B (ELO-included XGB) -- do trajectory features add anything once ELO is already present?",
}
diff_rows = []
for key, label in diff_labels.items():
    arr = np.array(boot_diff[key])
    lo, hi = np.percentile(arr, [2.5, 97.5])
    significant = not (lo < 0 < hi)
    direction = 'lower Brier (BETTER)' if arr.mean() < 0 else 'higher Brier (WORSE)'
    diff_rows.append({'comparison': key, 'description': label, 'mean_diff': arr.mean(),
                       'ci_lo': lo, 'ci_hi': hi, 'significant_95pct': significant, 'direction_of_mean': direction})
diff_table = pd.DataFrame(diff_rows).set_index('comparison')
diff_table_display = diff_table.copy()
for c in ['mean_diff', 'ci_lo', 'ci_hi']:
    diff_table_display[c] = diff_table_display[c].round(5)
diff_table_display[['description', 'mean_diff', 'ci_lo', 'ci_hi', 'significant_95pct', 'direction_of_mean']]

In [ ]:
print('Verdicts (95% paired bootstrap CI on Brier differences, primary held-out TEST set):\n')
for key, label in diff_labels.items():
    r = diff_table.loc[key]
    verdict = 'YES -- statistically significant' if r['significant_95pct'] else 'NOT statistically significant (CI includes 0)'
    print(label)
    print(f"  mean diff = {r['mean_diff']:+.5f}   95% CI = [{r['ci_lo']:+.5f}, {r['ci_hi']:+.5f}]   -> {verdict}\n")

# Bonus context (not a required deliverable comparison, but directly answers "did A' flip the sign vs ELO"):
a_prime_brier = results.loc[results['experiment'] == "Exp A' - ELO-free XGB + trajectory", 'brier'].values[0]
elo_brier_v = results.loc[results['experiment'] == 'Exp 0 - ELO baseline', 'brier'].values[0]
flip_msg = "A' now NUMERICALLY beats plain ELO" if a_prime_brier < elo_brier_v else "A' still behind plain ELO"
print(f"Bonus context -- Exp A' point-estimate Brier ({a_prime_brier:.5f}) vs Exp 0 ELO baseline "
      f"({elo_brier_v:.5f}): {flip_msg} "
      f"(point estimate only -- not bootstrap-tested here since it was not part of this notebook's primary comparison set).")

## Step 9 -- Feature importance: do the new trajectory features rank high?

In [ ]:
imp_a = pd.Series(model_a_prime_raw.feature_importances_, index=EXP_A_PRIME_COLS).sort_values(ascending=False)
imp_b = pd.Series(model_b_prime_raw.feature_importances_, index=EXP_B_PRIME_COLS).sort_values(ascending=False)

print("Exp A' feature importance (top 15 of 36):")
print(imp_a.head(15))
print("\nExp B' feature importance (top 15 of 37):")
print(imp_b.head(15))

In [ ]:
print("Trajectory feature ranks in Exp A' (ELO-free + trajectory, 36 total features):")
for c in TRAJ_DIFF_COLS:
    rank = list(imp_a.index).index(c) + 1
    print(f"  {c:35s} rank {rank:2d}/{len(imp_a)}   importance={imp_a[c]:.4f}")

print("\nTrajectory feature ranks in Exp B' (ELO-included + trajectory, 37 total features):")
for c in TRAJ_DIFF_COLS:
    rank = list(imp_b.index).index(c) + 1
    print(f"  {c:35s} rank {rank:2d}/{len(imp_b)}   importance={imp_b[c]:.4f}")

n_traj_top10_a = sum(1 for c in TRAJ_DIFF_COLS if list(imp_a.index).index(c) < 10)
n_traj_top10_b = sum(1 for c in TRAJ_DIFF_COLS if list(imp_b.index).index(c) < 10)
print(f"\n{n_traj_top10_a}/9 trajectory features rank in Exp A''s top 10 (of 36 total features).")
print(f"{n_traj_top10_b}/9 trajectory features rank in Exp B''s top 10 (of 37 total features).")

## Step 10 -- Save artifacts (gitignored: `../models/*.pkl`, `../data/*.parquet`)

In [ ]:
with open(MODELS_DIR / 'exp_a_prime_model.pkl', 'wb') as f:
    pickle.dump({'raw_model': model_a_prime_raw, 'calibrated_model': model_a_prime_cal,
                 'feature_cols': EXP_A_PRIME_COLS, 'best_params': BEST_PARAMS_A,
                 'trajectory_cols': TRAJ_DIFF_COLS}, f)

with open(MODELS_DIR / 'exp_b_prime_model.pkl', 'wb') as f:
    pickle.dump({'raw_model': model_b_prime_raw, 'calibrated_model': model_b_prime_cal,
                 'feature_cols': EXP_B_PRIME_COLS, 'best_params': BEST_PARAMS_B,
                 'trajectory_cols': TRAJ_DIFF_COLS}, f)

test_predictions_prime = test_df[['SusretId', 'Datum', 'SezonaId', 'LigaId', 'PlayerId', 'OpponentId',
                                    'Label', 'Perspective']].copy()
test_predictions_prime['p_expAprime_calibrated'] = p_test_a_prime
test_predictions_prime['p_expBprime_calibrated'] = p_test_b_prime
test_predictions_prime.to_parquet(DATA_DIR / 'test_predictions_trajectory.parquet', index=False)

print(f"Saved: {MODELS_DIR / 'exp_a_prime_model.pkl'}")
print(f"Saved: {MODELS_DIR / 'exp_b_prime_model.pkl'}")
print(f"Saved: {DATA_DIR / 'test_predictions_trajectory.parquet'}  (shape={test_predictions_prime.shape})")
lap("DONE")

## Summary -- Honest Bottom Line

**Comparison table (calibrated test Brier, held-out 6,852-row / 3,426-match TEST set,
2025-01-19..2026-07-08):**

| Experiment | Brier | vs ELO |
|---|---|---|
| Exp 0 -- ELO baseline | 0.17713 | -- |
| Exp A -- ELO-free XGB (no trajectory) | 0.19690 | +11.16% (worse) |
| **Exp A' -- ELO-free XGB + trajectory** | **0.17542** | **-0.97% (now beats plain ELO)** |
| Exp B -- ELO-included XGB | 0.16699 | -5.73% |
| **Exp B' -- ELO-included XGB + trajectory** | **0.16545** | **-6.60%** |
| Exp C -- ELO-free GRU | 0.16278 | -8.11% |

(Exact numbers reproduced by this notebook's own `results` table above -- this markdown table is
the pre-validated reference from the standalone script run.)

**New trajectory features (9 total, all vectorized pandas, no per-row loops, no walk-forward CV):**
`razina_ago5/10/20_diff`, `razina_slope10_diff`, `razina_stint_diff`, `razina_promotion_signal_diff`,
`razina_best_ever_diff`, `razina_current_minus_best_diff`, `forma_slope10_diff` (bonus).

**Paired-bootstrap verdicts (2,000 match-level resamples, seed 42):**
- **A' vs A: mean Brier diff ~ -0.0215, 95% CI excludes 0 -- highly significant.** Handing
  XGBoost the same razina-trajectory information the GRU sees, even as hand-engineered static
  summaries, produces a large, statistically bulletproof improvement. Exp A' also now
  numerically edges out the plain ELO formula itself (0.1754 vs 0.1771) -- something Exp A never
  came close to.
- **A' vs C: mean Brier diff ~ +0.0126, 95% CI excludes 0 -- still significant, but the gap
  shrank by roughly two-thirds** (from Exp A's +0.0341 vs C down to Exp A''s +0.0126 vs C --
  i.e. hand-engineered trajectory features closed ~63% of the original gap, but a real,
  statistically detectable remainder is left on the table for the GRU).
- **B' vs B: mean Brier diff ~ -0.0016, 95% CI includes 0 -- NOT significant.** Once ELO is
  already a feature, adding the same trajectory features gives no statistically detectable
  further improvement -- consistent with ELO already encoding most of the opponent-quality/level
  signal that the trajectory features re-derive from scratch for the ELO-free case.

**Feature importance -- yes, trajectory features rank high.** In Exp A', `razina_ago5_diff` is
literally the single most important feature in the entire 36-feature model (ahead of
`career_winrate_before_diff`, `debut_razina_diff`, and every other existing ELO-substitute
feature) -- see the ranked list above for the exact count in the top 10.

**Honest bottom line.** The GRU's edge over ELO-free XGBoost was NOT purely an artifact of "the
GRU sees razina as a trajectory and XGBoost didn't get the chance" -- when XGBoost IS given that
chance (via hand-engineered summary features), it closes roughly two-thirds of the gap and, for
the first time, produces an ELO-free tabular model that is competitive with the plain ELO formula
itself. But a real, statistically significant remainder of the GRU's edge survives even after
this fix (Brier 0.1754 vs 0.1628, CI excludes 0). The most defensible reading: hand-engineered
snapshots (5/10/20 matches ago, a linear slope, a stint counter, a promotion flag) are a lossy
compression of the full sequence shape that a recurrent network can learn to exploit more
completely -- shapes like "razina has been oscillating between two adjacent levels for the last 6
matches" or "the player just had their single best run of form in years, right before a
relegation" are visible to a GRU's learned hidden state but not to an a-priori-chosen set of 9
scalar summaries. So the GRU's edge is MOSTLY (but not entirely) explained by the trajectory
information itself, with a smaller residual advantage attributable to genuinely more flexible
(sequence-model-learned, vs. hand-specified) processing of that same information. This is
consistent with -- not a contradiction of -- `05`'s Step 15 ablation finding that razina
trajectory is the dominant mechanism: dominant does not mean exclusive, and this notebook is the
first place that remaining, non-trajectory portion of the GRU's edge is directly quantified and
shown to still be statistically real.

## Update: What Happened Next (08, 09)

*(Added after `08_gru_elo.ipynb` and `09_walkforward_top_models.ipynb` were built.)*

This notebook closed roughly two-thirds of Exp C's ELO-free edge by giving XGBoost the same razina trajectory as hand-engineered features (Exp A'), and found the remaining GRU advantage small but real. Two follow-ups completed the picture:

- **`08_gru_elo.ipynb`** built the GRU-side mirror of Exp B': adding point-in-time ELO to the GRU itself (Exp C2, not just to XGBoost). It found the exact same pattern in reverse — adding ELO to an already-strong ELO-free model (the GRU) gives no statistically significant lift either (Exp C2 vs Exp C: 95% CI on the Brier difference straddles 0). Combined with this notebook's B' vs B result, the two-directional finding is: **ELO and razina-trajectory (hand-engineered OR learned) are close substitutes for this dataset, each already capturing most of what the other would add.** Exp C2 does, however, significantly beat both Exp B and Exp B' — the best point estimate of every experiment in the project.
- **`09_walkforward_top_models.ipynb`** (capstone) walk-forward tests Exp B' and Exp C2 across 4 expanding time folds. Exp B' wins the earliest/smallest-data fold outright (the single best model of all four tested there) but loses to Exp C2 in the 3 later, larger-data folds — a concrete demonstration that this notebook's hand-engineered trajectory features are more sample-efficient early on, while the GRU's learned representation pulls ahead as more sequence history accumulates. See `../README.md` for the full results table and narrative.
